# Gold Layer: Star Schema Modelling

## Overview
This notebook implements the Gold layer of the Medallion Architecture, as described in Section 
3.9 of the dissertation. The cleansed Silver layer dataset is modelled into a star schema 
following the design proposed by Kimball and Ross (2013), consisting of a central 
Fact_Prescriptions table and two dimension tables: Dim_Provider and Dim_Drug. Before the 
dimension and fact tables are constructed, the Silver dataset is inspected to confirm the 
uniqueness and integrity of the candidate keys that will link the fact table to each dimension.

## Step 1: Load Silver Layer and Assess Candidate Keys

The cleansed dataset is loaded from the Silver container. The candidate key for Dim_Provider 
(Prscrbr_NPI) and the candidate key for Dim_Drug (the combination of Brnd_Name and Gnrc_Name) 
are examined to confirm how many distinct providers and distinct drugs exist in the dataset, and 
whether a single provider or drug combination is associated with more than one descriptive 
attribute value, which would need to be resolved before the dimension tables are built.

In [0]:
silver_path = "abfss://silver@adlspharmaanalytics.dfs.core.windows.net/prescribers_cleansed"
df_silver = spark.read.format("delta").load(silver_path)

# Distinct counts for the candidate dimension keys
distinct_providers = df_silver.select("Prscrbr_NPI").distinct().count()
distinct_drugs = df_silver.select("Brnd_Name", "Gnrc_Name").distinct().count()

print(f"Total rows in Silver: {df_silver.count():,}")
print(f"Distinct providers (Prscrbr_NPI): {distinct_providers:,}")
print(f"Distinct drug combinations (Brnd_Name + Gnrc_Name): {distinct_drugs:,}")

# Check whether a single NPI maps to more than one set of descriptive attributes (name, city, etc.)
provider_attribute_check = (
    df_silver.select("Prscrbr_NPI", "Prscrbr_Last_Org_Name", "Prscrbr_First_Name",
                      "Prscrbr_City", "Prscrbr_State_Abrvtn", "Prscrbr_Type")
    .distinct()
    .groupBy("Prscrbr_NPI")
    .count()
    .filter("count > 1")
)
inconsistent_provider_count = provider_attribute_check.count()
print(f"Providers with inconsistent attributes across rows: {inconsistent_provider_count:,}")

Total rows in Silver: 28,023,892
Distinct providers (Prscrbr_NPI): 1,139,455
Distinct drug combinations (Brnd_Name + Gnrc_Name): 3,159
Providers with inconsistent attributes across rows: 0


## Step 2: Build the Dim_Provider Dimension Table

With no inconsistencies found in provider attributes, Dim_Provider is constructed by selecting 
the distinct set of provider-level attributes, keyed on Prscrbr_NPI. This table will be joined to 
Fact_Prescriptions to support provider-level analysis such as prescribing volume by specialty, 
city, or state.

In [0]:
dim_provider = (
    df_silver
    .select("Prscrbr_NPI", "Prscrbr_Last_Org_Name", "Prscrbr_First_Name",
            "Prscrbr_City", "Prscrbr_State_Abrvtn", "Prscrbr_State_FIPS",
            "Prscrbr_Type", "Prscrbr_Type_Src")
    .distinct()
)

print(f"Dim_Provider row count: {dim_provider.count():,}")
display(dim_provider.limit(10))

# Write to Gold container
gold_path_provider = "abfss://gold@adlspharmaanalytics.dfs.core.windows.net/dim_provider"
dim_provider.write.format("delta").mode("overwrite").save(gold_path_provider)
print("Dim_Provider written to Gold layer.")

Dim_Provider row count: 1,139,455


Prscrbr_NPI,Prscrbr_Last_Org_Name,Prscrbr_First_Name,Prscrbr_City,Prscrbr_State_Abrvtn,Prscrbr_State_FIPS,Prscrbr_Type,Prscrbr_Type_Src
1598450058,Taylor,Chauncey,Hillsborough,NJ,34,Nurse Practitioner,Claim-Specialty
1598460206,Mehta,Jaya,New Orleans,LA,22,Student in an Organized Health Care Education/Training Program,NPPES-Taxonomy
1598460636,Somerville,Laura,Holyoke,MA,25,Nurse Practitioner,Claim-Specialty
1598446858,Nocera,Thomas,Santa Rosa,CA,06,Nurse Practitioner,NPPES-Specialty
1598441503,Frans,Mikali,Surprise,AZ,04,Nurse Practitioner,Claim-Specialty
1598447716,Earles,Kevin,Columbus,OH,39,Physician Assistant,Claim-Specialty
1598456675,Perry,Anastasia,North Wales,PA,42,Optometry,Claim-Specialty
1598458853,Spencer,Renee,Winder,GA,13,Nurse Practitioner,Claim-Specialty
1598450934,Osemwengie,Osadebamwen,Arlington,TX,48,Dentist,NPPES-Specialty
1598436016,Powers,Brenna,Saint Joseph,MI,26,Physician Assistant,Claim-Specialty


Dim_Provider written to Gold layer.


## Step 3: Build the Dim_Drug Dimension Table

Dim_Drug is constructed from the distinct combinations of Brnd_Name and Gnrc_Name identified in 
Step 1. As the source dataset does not provide a natural drug identifier, a surrogate key 
(Drug_Key) is generated for each distinct drug combination to serve as the join key with 
Fact_Prescriptions.

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

dim_drug = (
    df_silver
    .select("Brnd_Name", "Gnrc_Name")
    .distinct()
    .withColumn("Drug_Key", monotonically_increasing_id())
    .select("Drug_Key", "Brnd_Name", "Gnrc_Name")
)

print(f"Dim_Drug row count: {dim_drug.count():,}")
display(dim_drug.limit(10))

# Write to Gold container
gold_path_drug = "abfss://gold@adlspharmaanalytics.dfs.core.windows.net/dim_drug"
dim_drug.write.format("delta").mode("overwrite").save(gold_path_drug)
print("Dim_Drug written to Gold layer.")

Dim_Drug row count: 3,159


Drug_Key,Brnd_Name,Gnrc_Name
0,Ibuprofen,Ibuprofen
1,Guanfacine Hcl Er,Guanfacine Hcl
2,Fetroja,Cefiderocol Sulfate Tosylate
3,Azelastine Hcl,Azelastine Hcl
4,Varenicline Tartrate,Varenicline Tartrate
5,Alphagan P,Brimonidine Tartrate
6,Veo Insulin Syringe,"Syringe And Needle,insulin,1ml"
7,Victoza 2-Pak,Liraglutide
8,Fycompa,Perampanel
9,Humalog,Insulin Lispro


Dim_Drug written to Gold layer.


## Step 4: Build the Fact_Prescriptions Table

Fact_Prescriptions is constructed by joining the Silver layer dataset with Dim_Drug on the 
Brnd_Name and Gnrc_Name attributes, so that each row references the corresponding Drug_Key 
rather than repeating the descriptive drug attributes. Prscrbr_NPI is retained directly as the 
foreign key to Dim_Provider, since it already serves as the natural key for that dimension. The 
resulting fact table retains only the foreign keys and the measurable metrics for each 
prescription record, following the principle that fact tables should store quantitative measures 
and keys rather than descriptive attributes (Kimball and Ross, 2013).

In [0]:
fact_prescriptions = (
    df_silver
    .join(dim_drug, on=["Brnd_Name", "Gnrc_Name"], how="left")
    .select(
        "Prscrbr_NPI",
        "Drug_Key",
        "Tot_Clms",
        "Tot_30day_Fills",
        "Tot_Day_Suply",
        "Tot_Drug_Cst",
        "Tot_Benes",
        "GE65_Tot_Clms",
        "GE65_Tot_30day_Fills",
        "GE65_Tot_Drug_Cst",
        "GE65_Tot_Day_Suply",
        "GE65_Tot_Benes",
        "GE65_Sprsn_Reason",
        "GE65_Bene_Sprsn_Reason"
    )
)

print(f"Fact_Prescriptions row count: {fact_prescriptions.count():,}")
display(fact_prescriptions.limit(10))

# Sanity check: row count should match the Silver layer (28,023,892), confirming the join did not duplicate or drop rows
print(f"Silver layer row count for comparison: {df_silver.count():,}")

# Write to Gold container
gold_path_fact = "abfss://gold@adlspharmaanalytics.dfs.core.windows.net/fact_prescriptions"
fact_prescriptions.write.format("delta").mode("overwrite").save(gold_path_fact)
print("Fact_Prescriptions written to Gold layer.")

Fact_Prescriptions row count: 28,023,892


Prscrbr_NPI,Drug_Key,Tot_Clms,Tot_30day_Fills,Tot_Day_Suply,Tot_Drug_Cst,Tot_Benes,GE65_Tot_Clms,GE65_Tot_30day_Fills,GE65_Tot_Drug_Cst,GE65_Tot_Day_Suply,GE65_Tot_Benes,GE65_Sprsn_Reason,GE65_Bene_Sprsn_Reason
1194780320,67,21,35.0,1049,901.91,null,null,null,null,null,null,Direct Suppression,Direct Suppression
1194780320,114,17,39.0,1143,3247.19,null,null,null,null,null,null,Direct Suppression,Direct Suppression
1194780320,449,24,39.2,1046,4965.14,null,null,null,null,null,null,Counter Suppression,Direct Suppression
1194780320,868,24,52.1,1540,5080.12,null,24,52.1,5080.12,1540,null,Not Suppressed,Direct Suppression
1194780320,989,130,340.6,10070,16297.75,49,null,null,null,null,null,Counter Suppression,Counter Suppression
1194780320,1634,108,259.8,7794,1135.6,23,108,259.8,1135.6,7794,23,Not Suppressed,Not Suppressed
1194780320,1959,71,149.6,4461,67634.31,17,71,149.6,67634.31,4461,17,Not Suppressed,Not Suppressed
1194780320,2067,11,12.6,350,1873.98,null,11,12.6,1873.98,350,null,Not Suppressed,Direct Suppression
1194780320,2448,21,43.0,1290,24141.16,null,21,43.0,24141.16,1290,null,Not Suppressed,Direct Suppression
1194780320,2453,115,305.4,9144,24940.71,39,null,null,null,null,null,Counter Suppression,Counter Suppression


Silver layer row count for comparison: 28,023,892
Fact_Prescriptions written to Gold layer.


## Step 5: Referential Integrity Check

Before the star schema is considered complete, a validation join is performed to confirm that 
every Prscrbr_NPI and Drug_Key in Fact_Prescriptions has a matching record in Dim_Provider and 
Dim_Drug respectively. This ensures the relationships required for Power BI reporting will 
resolve correctly.

In [0]:
# Check for fact rows with no matching provider
orphan_providers = fact_prescriptions.join(dim_provider, on="Prscrbr_NPI", how="left_anti").count()

# Check for fact rows with no matching drug
orphan_drugs = fact_prescriptions.join(dim_drug, on="Drug_Key", how="left_anti").count()

print(f"Fact rows with no matching Dim_Provider record: {orphan_providers:,}")
print(f"Fact rows with no matching Dim_Drug record: {orphan_drugs:,}")

Fact rows with no matching Dim_Provider record: 0
Fact rows with no matching Dim_Drug record: 0


## Step 6: Register Gold Layer Tables in Unity Catalog

To enable Power BI to query the star schema via the Databricks Serverless SQL Warehouse using 
DirectQuery, as specified in Section 3.9, the three Delta tables constructed above are registered 
as managed references in Unity Catalog under a dedicated gold schema.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_pharma_analytics.gold;

CREATE TABLE IF NOT EXISTS adb_pharma_analytics.gold.dim_provider
USING DELTA
LOCATION 'abfss://gold@adlspharmaanalytics.dfs.core.windows.net/dim_provider';

CREATE TABLE IF NOT EXISTS adb_pharma_analytics.gold.dim_drug
USING DELTA
LOCATION 'abfss://gold@adlspharmaanalytics.dfs.core.windows.net/dim_drug';

CREATE TABLE IF NOT EXISTS adb_pharma_analytics.gold.fact_prescriptions
USING DELTA
LOCATION 'abfss://gold@adlspharmaanalytics.dfs.core.windows.net/fact_prescriptions';

SHOW TABLES IN adb_pharma_analytics.gold;

database,tableName,isTemporary
gold,dim_drug,false
gold,dim_provider,false
gold,fact_prescriptions,false


## Step 7: Gold Layer Analytical Views

To support the evaluation of analytical usefulness required by the fourth research objective, 
a set of aggregated SQL views is created on top of the star schema. These views address three 
areas: (i) the extent and nature of data suppression across the dataset, providing direct 
evidence for the third research question on pipeline efficiency; (ii) descriptive analytical 
outputs (prescribing volume and cost by state, specialty, and drug) demonstrating the analytical 
readiness of the Gold layer; and (iii) a comparison between the elderly (65 and over) and overall 
patient population, given that this subgroup is most affected by suppression.

In [0]:
%sql
CREATE OR REPLACE VIEW adb_pharma_analytics.gold.vw_suppression_summary AS
SELECT
    GE65_Sprsn_Reason,
    COUNT(*) AS record_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM adb_pharma_analytics.gold.fact_prescriptions), 2) AS percentage
FROM adb_pharma_analytics.gold.fact_prescriptions
GROUP BY GE65_Sprsn_Reason
ORDER BY record_count DESC;

SELECT * FROM adb_pharma_analytics.gold.vw_suppression_summary;

GE65_Sprsn_Reason,record_count,percentage
Not Suppressed,15560390,55.53
Counter Suppression,9374102,33.45
Direct Suppression,3089400,11.02


In [0]:
%sql
CREATE OR REPLACE VIEW adb_pharma_analytics.gold.vw_top_drugs AS
SELECT
    d.Gnrc_Name,
    SUM(f.Tot_Clms) AS total_claims,
    ROUND(SUM(f.Tot_Drug_Cst), 2) AS total_cost,
    COUNT(DISTINCT f.Prscrbr_NPI) AS distinct_prescribers
FROM adb_pharma_analytics.gold.fact_prescriptions f
JOIN adb_pharma_analytics.gold.dim_drug d ON f.Drug_Key = d.Drug_Key
GROUP BY d.Gnrc_Name
ORDER BY total_claims DESC;

SELECT * FROM adb_pharma_analytics.gold.vw_top_drugs;

Gnrc_Name,total_claims,total_cost,distinct_prescribers
"Insulin Glargine,hum.Rec.Anlog",72477208,7.9505564302E8,372496
Bortezomib,49241894,3.1075619443E8,343039
Benzoyl Peroxide,42465073,4.5634666535E8,284341
Carbidopa/Levodopa,35534666,5.4002657314E8,370346
Sevelamer Hcl,35418855,2.4813586655E8,309723
Ibuprofen,30104870,4.2652995348E8,262118
Meperidine Hcl,24446801,3.0877347172E8,301374
Pimecrolimus,22499877,1.1497303711E8,283173
Elviteg/Cob/Emtri/Tenof Alafen,19057273,3.7711253263E8,225916
Somatropin,17758956,7.839675411E7,247625


In [0]:
%sql
CREATE OR REPLACE VIEW adb_pharma_analytics.gold.vw_state_summary AS
SELECT
    p.Prscrbr_State_Abrvtn,
    SUM(f.Tot_Clms) AS total_claims,
    ROUND(SUM(f.Tot_Drug_Cst), 2) AS total_cost,
    COUNT(DISTINCT p.Prscrbr_NPI) AS distinct_prescribers
FROM adb_pharma_analytics.gold.fact_prescriptions f
JOIN adb_pharma_analytics.gold.dim_provider p ON f.Prscrbr_NPI = p.Prscrbr_NPI
GROUP BY p.Prscrbr_State_Abrvtn
ORDER BY total_cost DESC;

SELECT * FROM adb_pharma_analytics.gold.vw_state_summary;

Prscrbr_State_Abrvtn,total_claims,total_cost,distinct_prescribers
CA,129565592,2.126240016458E10,113975
NY,95240368,1.817413128085E10,79262
FL,113582265,1.683893791776E10,78092
TX,98638359,1.574644193372E10,75623
PA,72980145,1.11283471404E10,52365
OH,62294899,9.04307142401E9,44830
NC,52430568,8.1382153483E9,37687
MI,49623865,7.82134798871E9,38086
IL,51867241,7.53698273627E9,42228
GA,47568589,7.3041295703E9,30920


In [0]:
%sql
CREATE OR REPLACE VIEW adb_pharma_analytics.gold.vw_specialty_summary AS
SELECT
    p.Prscrbr_Type,
    SUM(f.Tot_Clms) AS total_claims,
    ROUND(SUM(f.Tot_Drug_Cst), 2) AS total_cost,
    COUNT(DISTINCT p.Prscrbr_NPI) AS distinct_prescribers
FROM adb_pharma_analytics.gold.fact_prescriptions f
JOIN adb_pharma_analytics.gold.dim_provider p ON f.Prscrbr_NPI = p.Prscrbr_NPI
GROUP BY p.Prscrbr_Type
ORDER BY total_cost DESC;

SELECT * FROM adb_pharma_analytics.gold.vw_specialty_summary;

Prscrbr_Type,total_claims,total_cost,distinct_prescribers
Nurse Practitioner,269894701,3.747637729257E10,229976
Internal Medicine,332609911,3.355009252182E10,104182
Family Practice,371050213,2.917441433273E10,105120
Hematology-Oncology,7083076,1.72889310259E10,8746
Physician Assistant,95064409,1.355894645344E10,114276
Cardiology,64689193,1.349355991414E10,18782
Pulmonary Disease,13915148,8.39962585917E9,9233
Rheumatology,10797149,8.39883929358E9,5224
Neurology,20245985,7.9868674276E9,13695
Endocrinology,17972894,7.96957586253E9,6527


In [0]:
%sql
CREATE OR REPLACE VIEW adb_pharma_analytics.gold.vw_elderly_vs_overall AS
SELECT
    SUM(Tot_Clms) AS overall_total_claims,
    SUM(GE65_Tot_Clms) AS elderly_total_claims,
    ROUND(SUM(GE65_Tot_Clms) * 100.0 / SUM(Tot_Clms), 2) AS elderly_claims_pct_of_known,
    SUM(Tot_Drug_Cst) AS overall_total_cost,
    SUM(GE65_Tot_Drug_Cst) AS elderly_total_cost,
    ROUND(SUM(GE65_Tot_Drug_Cst) * 100.0 / SUM(Tot_Drug_Cst), 2) AS elderly_cost_pct_of_known
FROM adb_pharma_analytics.gold.fact_prescriptions;

SELECT * FROM adb_pharma_analytics.gold.vw_elderly_vs_overall;

overall_total_claims,elderly_total_claims,elderly_claims_pct_of_known,overall_total_cost,elderly_total_cost,elderly_cost_pct_of_known
1479628807,815092470,55.09,2.2674090213058997E11,1.277585079458392E11,56.35
